# Simple Sanity Check

This notebook checks the raw CSVs in `data/01_raw/` before running the rest of the pipeline.

In [ ]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("data/01_raw")

print("Raw data folder:", RAW_DIR.resolve())
print("Exists:", RAW_DIR.exists())

In [ ]:
required_files = [
    "customer_demographics.csv",
    "product_propensity.csv",
    "product_holdings.csv",
    "transaction_aggregates.csv",
    "raw_transactions.csv",
    "digital_login.csv",
    "digital_clicks.csv",
    "product_metadata.csv",
    "conversion_data_july_2026.csv",
]

overview = []

for file in required_files:
    path = RAW_DIR / file
    if path.exists():
        df = pd.read_csv(path)
        overview.append({
            "file": file,
            "status": "FOUND",
            "rows": len(df),
            "columns": len(df.columns),
        })
    else:
        overview.append({
            "file": file,
            "status": "MISSING",
            "rows": None,
            "columns": None,
        })

overview_df = pd.DataFrame(overview)
display(overview_df)

In [ ]:
# Load main files
customer_demographics = pd.read_csv(RAW_DIR / "customer_demographics.csv")
product_propensity = pd.read_csv(RAW_DIR / "product_propensity.csv")
product_holdings = pd.read_csv(RAW_DIR / "product_holdings.csv")
transaction_aggregates = pd.read_csv(RAW_DIR / "transaction_aggregates.csv")
raw_transactions = pd.read_csv(RAW_DIR / "raw_transactions.csv")
digital_login = pd.read_csv(RAW_DIR / "digital_login.csv")
digital_clicks = pd.read_csv(RAW_DIR / "digital_clicks.csv")
product_metadata = pd.read_csv(RAW_DIR / "product_metadata.csv")

print("Files loaded successfully.")

In [ ]:
# Customer ID linkage check
base_customers = set(customer_demographics["Customer ID"])

checks = []

table_customer_cols = {
    "product_propensity": (product_propensity, "Customer ID"),
    "product_holdings": (product_holdings, "Customer ID"),
    "transaction_aggregates": (transaction_aggregates, "CustID"),
    "raw_transactions": (raw_transactions, "CustID"),
    "digital_login": (digital_login, "CustID"),
    "digital_clicks": (digital_clicks, "CustID"),
}

for table_name, (df, customer_col) in table_customer_cols.items():
    table_customers = set(df[customer_col])
    checks.append({
        "table": table_name,
        "unique_customers": len(table_customers),
        "customers_not_in_demographics": len(table_customers - base_customers),
    })

display(pd.DataFrame(checks))

In [ ]:
# Product holding sanity check
holding_cols = {
    "PL": "PL_count",
    "CC": "CC_count",
    "HL": "HL_count",
    "SA": "SA_count",
    "RD": "RD_count",
    "MF": "MF_count",
}

holding_rates = []

for product, col in holding_cols.items():
    holding_rates.append({
        "product": product,
        "holding_rate": (product_holdings[col] > 0).mean()
    })

holding_rates_df = pd.DataFrame(holding_rates).sort_values("holding_rate", ascending=False)
display(holding_rates_df)

print("Expected pattern: SA should be highest, CC should be much higher than HL, and HL should be relatively low.")

In [ ]:
# Propensity score range check
propensity_cols = ["PL_P", "CC_P", "HL_P", "SA_P", "RD_P", "MF_P"]

propensity_checks = []

for col in propensity_cols:
    propensity_checks.append({
        "column": col,
        "min": product_propensity[col].min(),
        "max": product_propensity[col].max(),
        "outside_0_1": ((product_propensity[col] < 0) | (product_propensity[col] > 1)).sum(),
    })

display(pd.DataFrame(propensity_checks))

In [ ]:
# Basic final sanity status
all_files_found = overview_df["status"].eq("FOUND").all()
no_bad_customer_links = pd.DataFrame(checks)["customers_not_in_demographics"].sum() == 0
all_propensity_valid = pd.DataFrame(propensity_checks)["outside_0_1"].sum() == 0

print("All required files found:", all_files_found)
print("No invalid customer links:", no_bad_customer_links)
print("All propensity scores between 0 and 1:", all_propensity_valid)

if all_files_found and no_bad_customer_links and all_propensity_valid:
    print("PASS: Raw data is ready for the feature engineering pipeline.")
else:
    print("WARN: Check the issues above before running the pipeline.")